# Notebook 03 — Forecast PyCaret `time_series` sobre SAB.MC

**Objetivo:** entrenar un AutoML de forecast con PyCaret `time_series`, evaluarlo contra el mismo holdout de 90 días que usó el Prophet del notebook 02 y decidir cuál se queda como modelo de referencia.

**Pipeline:** `setup → compare_models → create_model → predict_model (holdout) → finalize_model → predict_model (futuro 90d) → save_model`.

**Tracking:** MLflow local en `./mlruns/`, experimento `sab_forecast_pycaret`.

**Nota sobre frecuencia:** la cotización diaria tiene huecos de fin de semana / festivos. PyCaret `time_series` necesita un índice con frecuencia regular, así que reindexamos a `'B'` (días hábiles) y rellenamos con `ffill`. Esto significa que los 90 puntos del holdout cubren ~4.5 meses naturales, no 90 días corridos como en el NB 02 — lo comentamos en la comparativa.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from pycaret.time_series import (
    setup, compare_models, create_model,
    finalize_model, predict_model, save_model,
    plot_model, pull
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

## 2. Carga del dataset limpio (salida del NB 01)

In [ ]:
sab = pd.read_csv(ROOT / "data" / "sab_5y_clean.csv", parse_dates=["Date"])
print("Filas:", len(sab))
print("Rango:", sab["Date"].min().date(), "->", sab["Date"].max().date())
sab.head(3)

## 3. Preparación de la serie univariante

PyCaret `time_series` espera una `pd.Series` (o `DataFrame` univariante) con índice temporal de frecuencia regular. Pasos:

1. Quedarse con `Date` y `Close`.
2. Indexar por fecha y forzar frecuencia `'B'` (business days) — rellena los días hábiles que falten.
3. `ffill` para arrastrar el último cierre conocido en huecos puntuales (festivos sueltos).

In [ ]:
ts = (
    sab[["Date", "Close"]]
      .rename(columns={"Date": "ds", "Close": "y"})
      .dropna()
      .set_index("ds")
      .asfreq("B")
      .ffill()
)["y"]

print("Tipo índice :", type(ts.index).__name__)
print("Frecuencia  :", ts.index.freqstr)
print("Filas       :", len(ts))
print("Rango       :", ts.index.min().date(), "->", ts.index.max().date())
ts.tail()

## 4. Configurar MLflow

Mismo tracking URI que el NB 02 para que todos los runs queden bajo `./mlruns/`. Cambiamos sólo el nombre de experimento.

In [ ]:
mlflow.set_tracking_uri('file:///' + (ROOT / 'mlruns').as_posix())
mlflow.set_experiment("sab_forecast_pycaret")

## 5. `setup()` — preparar el experimento

- `fh=90` → mismo horizonte que el NB 02 (90 días hábiles aquí).
- `fold=3` → CV walk-forward ligero (la serie no es enorme y queremos iterar rápido).
- `seasonal_period=5` → estacionalidad semanal bursátil (5 días hábiles).
- `numeric_imputation_target='ffill'` por consistencia con la serie.
- `log_experiment='mlflow'` → registra cada modelo del `compare_models` automáticamente.

In [ ]:
HORIZON = 90

s = setup(
    data=ts,
    fh=HORIZON,
    fold=3,
    seasonal_period=5,
    numeric_imputation_target="ffill",
    session_id=42,
    # log_experiment desactivado por incompatibilidad pycaret+mlflow3
    
    log_plots=False,   # los plots los hacemos a mano abajo
    verbose=False,
)
pull()

## 6. `compare_models()` — barrido AutoML

Excluimos `prophet` (ya está en el NB 02) y los modelos más pesados de redes neuronales para que la celda termine en tiempo razonable.

In [ ]:
best = compare_models(
    sort="MASE",
    n_select=1,
    exclude=["prophet", "bats", "tbats", "lar_cds_dt"],
    turbo=True,
    verbose=False,
)
leaderboard = pull()
leaderboard.head(10)

In [ ]:
print("Modelo ganador:")
print(best)

## 7. Evaluación en el holdout

`predict_model(best)` predice los `fh` puntos siguientes al final del train, que coinciden con el test reservado por PyCaret. Calculamos MAE/RMSE/MAPE igual que en el NB 02 para poder comparar.

In [ ]:
holdout_pred = predict_model(best)
holdout_pred.head()

In [ ]:
# y_pred sale como la primera (y única) columna numérica del DataFrame devuelto
y_pred = holdout_pred.iloc[:, 0]
# PyCaret devuelve PeriodIndex (freq B); lo pasamos a timestamp para alinear con ts
if hasattr(y_pred.index, "to_timestamp"):
    y_pred.index = y_pred.index.to_timestamp()
common_idx = ts.index.intersection(y_pred.index)
y_true = ts.loc[common_idx]
y_pred = y_pred.loc[common_idx]

mae_pc  = (y_true - y_pred).abs().mean()
rmse_pc = np.sqrt(((y_true - y_pred) ** 2).mean())
mape_pc = ((y_true - y_pred).abs() / y_true).mean() * 100

print(f"PyCaret ({type(best).__name__}) · MAE: {mae_pc:.3f}  RMSE: {rmse_pc:.3f}  MAPE: {mape_pc:.2f}%")

# Log manual a MLflow además del autologging de PyCaret
with mlflow.start_run(run_name="pycaret_best_holdout_metrics"):
    mlflow.log_param("model", type(best).__name__)
    mlflow.log_param("horizon", HORIZON)
    mlflow.log_param("freq", "B")
    mlflow.log_metric("mae",  mae_pc)
    mlflow.log_metric("rmse", rmse_pc)
    mlflow.log_metric("mape", mape_pc)

## 8. Comparativa con el Prophet del NB 02

Cargamos el forecast guardado por el NB 02 (`sab_forecast_prophet_90d.csv`) y lo evaluamos sobre el mismo tramo de fechas que el holdout de PyCaret, para tener una comparación lo más justa posible.

> ⚠️ La comparación no es 100% apples-to-apples: el NB 02 trabaja con freq `D` (90 días corridos) y aquí freq `B` (90 días hábiles ≈ 4.5 meses). Útil como referencia, no como veredicto definitivo.

In [ ]:
prophet_csv = ROOT / "data" / "sab_forecast_prophet_90d.csv"

if prophet_csv.exists():
    prophet_fcst = pd.read_csv(prophet_csv, parse_dates=["ds"]).set_index("ds")
    # Tomamos los puntos del forecast Prophet que caen en las fechas del holdout PyCaret
    common = prophet_fcst.index.intersection(y_pred.index)
    if len(common) > 0:
        y_pred_pr = prophet_fcst.loc[common, "yhat"]
        y_true_pr = ts.loc[common]
        mae_pr  = (y_true_pr - y_pred_pr).abs().mean()
        rmse_pr = np.sqrt(((y_true_pr - y_pred_pr) ** 2).mean())
        mape_pr = ((y_true_pr - y_pred_pr).abs() / y_true_pr).mean() * 100
    else:
        mae_pr = rmse_pr = mape_pr = np.nan
        print("Sin solapamiento de fechas entre Prophet y el holdout PyCaret.")
else:
    mae_pr = rmse_pr = mape_pr = np.nan
    print("No existe data/sab_forecast_prophet_90d.csv (ejecuta NB 02 primero).")

resumen = pd.DataFrame({
    "Modelo": [f"PyCaret ({type(best).__name__})", "Prophet (NB 02, con OPA holidays)"],
    "MAE":    [mae_pc, mae_pr],
    "RMSE":   [rmse_pc, rmse_pr],
    "MAPE %": [mape_pc, mape_pr],
}).set_index("Modelo")
resumen

## 9. Visualización del holdout

In [ ]:
train_tail = ts.iloc[-(HORIZON + 180):-HORIZON]  # últimos ~9 meses antes del holdout
test_real  = ts.iloc[-HORIZON:]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_tail.index, train_tail.values, color="black", label="Train (últimos 9m)", linewidth=1)
ax.plot(test_real.index,  test_real.values,  color="black", linestyle="--", label="Real (holdout)", linewidth=1.5)
ax.plot(y_pred.index,     y_pred.values,     color="tab:green",
        label=f"PyCaret {type(best).__name__} (RMSE {rmse_pc:.2f})")

if not np.isnan(rmse_pr):
    ax.plot(y_pred_pr.index, y_pred_pr.values, color="tab:blue",
            label=f"Prophet + OPA (RMSE {rmse_pr:.2f})", alpha=0.8)

ax.set_title("SAB.MC — Holdout 90 días: PyCaret vs Prophet")
ax.set_xlabel("Fecha"); ax.set_ylabel("EUR")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 10. Forecast 90 días al futuro

Reentrenamos con TODO el histórico (`finalize_model`) y proyectamos los próximos 90 días hábiles.

In [ ]:
final = finalize_model(best)
future_pred = predict_model(final, fh=HORIZON)
future_pred.head()

In [ ]:
y_future = future_pred.iloc[:, 0]
if hasattr(y_future.index, "to_timestamp"):
    y_future.index = y_future.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index, ts.values, color="black", label="Histórico", linewidth=1)
ax.plot(y_future.index, y_future.values, color="tab:green",
        label=f"Forecast 90d ({type(final).__name__})", linewidth=1.5)
ax.set_title("SAB.MC — Forecast 90 días al futuro (PyCaret time_series)")
ax.set_xlabel("Fecha"); ax.set_ylabel("EUR")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 11. Guardar modelo y previsión

In [ ]:
# Modelo
model_path = ROOT / "data" / "sab_pycaret_ts"   # PyCaret añade '.pkl' por su cuenta
save_model(final, str(model_path))
print(f"Modelo guardado en: {model_path}.pkl")

# Forecast a CSV (con índice como columna 'ds' para alinear con el NB 02)
fcst_out = y_future.rename("yhat").reset_index().rename(columns={"index": "ds"})
fcst_out_path = ROOT / "data" / "sab_forecast_pycaret_90d.csv"
fcst_out.to_csv(fcst_out_path, index=False)
print(f"Forecast guardado en: {fcst_out_path}")

## 12. Conclusiones

- Tenemos un segundo baseline serio (PyCaret AutoML) que podemos comparar contra el Prophet del NB 02.
- Si MAE/RMSE de PyCaret bajan del Prophet, lo promovemos a referencia y el Prophet queda como ablation. Si no, Prophet sigue siendo el modelo de cabecera y PyCaret aporta robustez al benchmark.
- Tracking en MLflow: `mlflow ui --backend-store-uri ./mlruns` → comparar runs de los dos experimentos (`sab_forecast_prophet` vs `sab_forecast_pycaret`).

**Siguiente paso natural:** `04_regresores_externos.ipynb` — añadir IBEX35, EUR/USD y tipos BCE como regresores exógenos al mejor modelo.